In [1]:
%pip install gplearn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import torch

# =========================================================
# 1. Environment Variables (Must be set early)
# =========================================================
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # Suppress TensorFlow C++ logs (0=all, 1=info, 2=warning, 3=error)
os.environ["TOKENIZERS_PARALLELISM"] = "false"     # Suppress Hugging Face parallelism warnings
os.environ["LOGURU_LEVEL"] = "ERROR"              # Suppress Loguru warnings
os.environ["PYTHONDEVMODE"] = "0"

# =========================================================
# 2. Python Standard Library Warnings
# =========================================================
warnings.filterwarnings("ignore")
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=DeprecationWarning)
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=SyntaxWarning)

# =========================================================
# 3. Pandas Configuration
# =========================================================
pd.options.mode.chained_assignment = None  # Suppress SettingWithCopyWarning
pd.set_option("mode.chained_assignment", None)

# =========================================================
# 4. NumPy Configuration
# =========================================================
# Suppress division by zero, overflow, underflow, and invalid value warnings
np.seterr(all="ignore")

# =========================================================
# 5. PyTorch Configuration
# =========================================================
torch.set_warn_always(False)

# Optional: Redirect C/C++ low-level stderr (e.g., CUDA / C++ driver output)
class SuppressStderr:
    def __enter__(self):
        self.errnull_file = open(os.devnull, 'w')
        self.old_stderr = sys.stderr
        sys.stderr = self.errnull_file
        return self

    def __exit__(self, *_):
        sys.stderr = self.old_stderr
        self.errnull_file.close()

# Example wrapper if you have stubborn C-level library warnings:
# with SuppressStderr():
#     model.fit(...)

In [3]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ndcg_score
import seaborn as sns
project_root = os.path.abspath('/kaggle/input/datasets/phmhuyphongp/modules-vn')
if project_root not in sys.path:
    sys.path.append(project_root)

import src.models as md
import src.evaluation as ev
import src.features as ft
from config import candidate_features, BASE_MODEL_PARAMS

# Data & Processing

In [4]:
def plot_equity_curves(*results, labels=None, normalize=False, regime_colors=None, show_regime = False):
    plt.figure(figsize=(14, 7))
    
    # 1. Bảng màu mặc định cho 4 Regimes (bạn có thể tùy chỉnh)
    if regime_colors is None:
        regime_colors = {
            'Q1': '#2ca02c',  # Xanh lá (vd: Tăng trưởng tốt)
            'Q2': '#1f77b4',  # Xanh dương (vd: Ổn định)
            'Q3': '#ff7f0e',  # Cam (vd: Cảnh báo / Biến động)
            'Q4': '#d62728'   # Đỏ (vd: Giảm mạnh / Suy thoái)
        }

    # 2. Xử lý tô màu nền Regime (Lấy theo DataFrame đầu tiên)
    if show_regime and len(results) > 0 and 'regime_bucket_monthly' in results[0].columns:
        df_regime = results[0].copy()
        df_regime['date'] = pd.to_datetime(df_regime['date'])
        df_regime = df_regime.sort_values('date').reset_index(drop=True)

        # Tìm các khoảng thời gian liên tục của từng regime
        # (Xác định điểm bắt đầu/thay đổi trạng thái)
        df_regime['regime_change'] = df_regime['regime_bucket_monthly'] != df_regime['regime_bucket_monthly'].shift(1)
        df_regime['group_id'] = df_regime['regime_change'].cumsum()

        added_regime_labels = set()

        for _, group in df_regime.groupby('group_id'):
            regime = group['regime_bucket_monthly'].iloc[0]
            start_date = group['date'].iloc[0]
            end_date = group['date'].iloc[-1]
            
            color = regime_colors.get(regime, '#cccccc') # Mặc định xám nếu không có trong dict
            
            # Chỉ thêm label vào legend 1 lần cho mỗi loại Regime
            reg_label = f'Regime {regime}' if regime not in added_regime_labels else None
            if reg_label:
                added_regime_labels.add(regime)

            # Tô màu nền cho khoảng thời gian này
            plt.axvspan(start_date, end_date, color=color, alpha=0.18, label=reg_label)

    # 3. Vẽ đường Equity Curve (Giữ nguyên logic của bạn)
    for i, result in enumerate(results):
        result = result.copy()
        result['date'] = pd.to_datetime(result['date'])
        label = labels[i] if labels and i < len(labels) else f'Strategy {i+1}'
        y = result['total_value']
        if normalize: 
            y = (y / y.iloc[0] - 1) * 100
            
        plt.plot(result['date'], y, linewidth=2, label=label, color='black' if len(results)==1 else None)

    # 4. Trang trí biểu đồ
    plt.title('Biểu đồ so sánh P1 và VNINDEX', fontsize=14, fontweight='bold')
    plt.xlabel('Ngày', fontsize=12)
    plt.ylabel('Lợi nhuận đã được chuẩn hóa (%)' if normalize else 'Tổng giá trị', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', framealpha=0.9)
    plt.tight_layout()
    plt.show()


In [5]:


def ndcg_at_20_series(final_df, eval_target_col, top_k=20):
    """Trả về pd.Series chứa NDCG@20 với index là date."""
    results = {}
    for date, g in final_df.groupby("date"):
        if len(g) > 1:
            # ndcg_score yêu cầu 2D array: [y_true], [y_score]
            score = ndcg_score([g[eval_target_col].values], [g["pred_score"].values], k=top_k)
            results[date] = score
    
    s = pd.Series(results, name="NDCG@20")
    s.index = pd.to_datetime(s.index) # Đảm bảo index ở dạng datetime để vẽ đường nối chuẩn xác
    return s.sort_index()

In [6]:
df_raw = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/qualified-stocks-for-researching/market_data_filtered.parquet')

df_predict_ndcg = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/nkkh-predicted-df/ndcg.parquet')

df_predict_mse = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/nkkh-predicted-df/mse.parquet')

df_predict_linear = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/nkkh-predicted-df/linear.parquet')
df_predict_lstm = pd.read_parquet('/kaggle/input/datasets/phmhuyphongp/nkkh-predicted-df/lstm.parquet')



from src.simulation import OrderManager, FilterGroup, RampFilter,StepFilter, build_data_engines

In [7]:
filter_p1 = FilterGroup(condition = "market1m_ema21-market3m_ema63<0 and regime_bucket_monthly =='Q1'",transformations = [StepFilter(0)])
filter_p2 = FilterGroup(condition = "regime_bucket_monthly =='Q1'",transformations = [StepFilter(0.5)])
filter_p3 = FilterGroup(condition = "market1m_ema21-market3m_ema63<0", transformations = [StepFilter(0.5)])

In [8]:
raw_engine_p1,predict_engine_p1,condition_engine_p1=build_data_engines(df_raw,df_predict_ndcg,df_predict_ndcg)
omp1 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p1, df_predict = predict_engine_p1,df_condition=condition_engine_p1,regime_filter=filter_p1)
omp1.run_strategy()
history_p1 = pd.DataFrame(omp1.portfolio.history, columns=["date", "total_value"])

raw_engine_p2,predict_engine_p2,condition_engine_p2=build_data_engines(df_raw,df_predict_ndcg,df_predict_ndcg)
omp2 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p2, df_predict = predict_engine_p2,df_condition=condition_engine_p2,regime_filter=filter_p2)
omp2.run_strategy()
history_p2 = pd.DataFrame(omp2.portfolio.history, columns=["date", "total_value"])

raw_engine_p3,predict_engine_p3,condition_engine_p3=build_data_engines(df_raw,df_predict_ndcg,df_predict_ndcg)
omp3 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p3, df_predict = predict_engine_p3,df_condition=condition_engine_p3,regime_filter=filter_p3)
omp3.run_strategy()
history_p3 = pd.DataFrame(omp3.portfolio.history, columns=["date", "total_value"])

raw_engine_p4,predict_engine_p4,condition_engine_p4=build_data_engines(df_raw,df_predict_ndcg,df_predict_ndcg)
omp4 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p4, df_predict = predict_engine_p4)
omp4.run_strategy()
history_p4 = pd.DataFrame(omp4.portfolio.history, columns=["date", "total_value"])

raw_engine_p5,predict_engine_p5,condition_engine_p5=build_data_engines(df_raw,df_predict_linear,df_predict_linear)
omp5 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p5, df_predict = predict_engine_p5)
omp5.run_strategy()
history_p5 = pd.DataFrame(omp5.portfolio.history, columns=["date", "total_value"])

raw_engine_p6,predict_engine_p6,condition_engine_p6=build_data_engines(df_raw,df_predict_mse,df_predict_mse)
omp6 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p6, df_predict = predict_engine_p6)
omp6.run_strategy()
history_p6 = pd.DataFrame(omp6.portfolio.history, columns=["date", "total_value"])

raw_engine_p7,predict_engine_p7,condition_engine_p7=build_data_engines(df_raw,df_predict_lstm,df_predict_lstm)
omp7 = OrderManager(initial = 500_000,topk=20,df_raw = raw_engine_p7, df_predict = predict_engine_p7)
omp7.run_strategy()
history_p7 = pd.DataFrame(omp7.portfolio.history, columns=["date", "total_value"])




# Q1:Model vs Model: P4, P5, P6, P7

In [9]:
# Tính toán chuỗi NDCG cho từng mô hình
s_p4 = ndcg_at_20_series(df_predict_ndcg, 'target_magnitude')
s_p6 = ndcg_at_20_series(df_predict_mse, 'target_magnitude')
s_p5 = ndcg_at_20_series(df_predict_linear, 'target_magnitude')
s_p7 = ndcg_at_20_series(df_predict_lstm, 'target_magnitude')


In [10]:
scores = {'p4(xgb ndcg)': s_p4, 'p6(xgb mse)': s_p6, 'p5(lassocv)': s_p5, 'p7(lstm)': s_p7}
for key, score in scores.items():
    print(f"{key}: {score.mean():.4f}")

p4(xgb ndcg): 0.6193
p6(xgb mse): 0.6019
p5(lassocv): 0.5931
p7(lstm): 0.6050


In [11]:
print(f"p4(NDCG): {ev.print_performance_report(history_p4, initial_capital=None, rf_annual=0.045, trading_days_per_year=252, verbose=False)['sharpe']:.4f}")
print(f"p5(LassoCV): {ev.print_performance_report(history_p5, initial_capital=None, rf_annual=0.045, trading_days_per_year=252, verbose=False)['sharpe']:.4f}")
print(f"p6(xgb mse): {ev.print_performance_report(history_p6, initial_capital=None, rf_annual=0.045, trading_days_per_year=252, verbose=False)['sharpe']:.4f}")
print(f"p7(lstm): {ev.print_performance_report(history_p7, initial_capital=None, rf_annual=0.045, trading_days_per_year=252, verbose=False)['sharpe']:.4f}")


p4(NDCG): 0.5491
p5(LassoCV): -0.0096
p6(xgb mse): 0.2389
p7(lstm): 0.3183


# Q2:

In [12]:
import pandas as pd

histories = {
    'p1': history_p1,
    'p2': history_p2,
    'p3': history_p3,
    'p4': history_p4,
}

rows = []
for key, hist in histories.items():
    perf = ev.print_performance_report(
        hist,
        initial_capital=None,
        rf_annual=0.045,
        trading_days_per_year=252,
        verbose=False
    )
    rows.append({
        'portfolio': key,
        'sharpe': perf['sharpe'],
        'max_drawdown': perf['max_drawdown']
    })

df_summary = pd.DataFrame(rows).set_index('portfolio')
df_summary_display = df_summary.applymap(lambda x: f"{x:.4f}")
print(df_summary_display)

           sharpe max_drawdown
portfolio                     
p1         0.9475      -0.3444
p2         0.7429      -0.4053
p3         0.3520      -0.4020
p4         0.5491      -0.5355


# Test cho std

Mã có độ biến động cao nhất:
DSC : 0.04387593390747919
Mã có độ biến động trung bình:
DSC : 0.02249573144510049
